# EEG Brain Signal Classifier — Step-by-Step Walkthrough

This notebook walks through the complete pipeline:
1. Generate mock EEG data
2. Preprocess with MNE-Python
3. Convert to spectrogram images
4. Train a CNN (ResNet18 with transfer learning)
5. Evaluate with metrics
6. Explain predictions with Grad-CAM

In [ ]:
# Install dependencies (run once)
# !pip install torch torchvision mne scipy matplotlib seaborn scikit-learn Pillow tqdm grad-cam

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Step 1 — Generate Mock EEG Data

We simulate 3-class EEG signals (focused / relaxed / stressed).
Replace this with real DEAP/SEED loading for actual research.

In [ ]:
from src.preprocess import generate_mock_eeg, LABEL_MAP

raw_list, labels = generate_mock_eeg(n_trials=30, duration_sec=30)
print(f'Generated {len(raw_list)} trials')
print(f'Label distribution: {dict(zip(*np.unique(labels, return_counts=True)))}')

## Step 2 — Visualise Raw EEG Signal

In [ ]:
raw = raw_list[0]
data, times = raw[:4, :512]  # first 4 channels, first 4 seconds

fig, axes = plt.subplots(4, 1, figsize=(12, 6), sharex=True)
for i, ax in enumerate(axes):
    ax.plot(times, data[i] * 1e6, linewidth=0.8)
    ax.set_ylabel(f'Ch {i+1} (µV)', fontsize=8)
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel('Time (s)')
fig.suptitle(f'Raw EEG — Label: {LABEL_MAP[labels[0]].upper()}', fontsize=12)
plt.tight_layout()
plt.show()

## Step 3 — Preprocess and Generate Spectrograms

In [ ]:
from src.preprocess import preprocess_raw, epoch_raw, epoch_to_spectrogram

# Preprocess one trial
raw_clean = preprocess_raw(raw_list[0])
epochs    = epoch_raw(raw_clean, epoch_duration=4.0)
print(f'Number of epochs: {len(epochs)}')

# Generate spectrogram for first epoch
ep_data = epochs.get_data()[0]  # shape: (channels, time)
spec_img = epoch_to_spectrogram(ep_data, sfreq=raw_clean.info['sfreq'])

plt.figure(figsize=(6, 4))
plt.imshow(spec_img)
plt.title('EEG Spectrogram (averaged across channels)')
plt.axis('off')
plt.tight_layout()
plt.show()

## Step 4 — Save All Spectrograms to Disk

In [ ]:
from src.preprocess import save_spectrograms

DATA_DIR = '../data/spectrograms'
save_spectrograms(raw_list, labels, output_dir=DATA_DIR)

## Step 5 — Load Dataset and Create DataLoaders

In [ ]:
from src.dataset import get_dataloaders

loaders, n_total = get_dataloaders(
    data_dir   = DATA_DIR,
    batch_size = 8,
    img_size   = 224,
)

# Visualise a batch
CLASSES = ['focused', 'relaxed', 'stressed']
imgs, lbls = next(iter(loaders['train']))
print(f'Batch shape: {imgs.shape}  Labels: {lbls.tolist()}')

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
mean = np.array([0.485, 0.456, 0.406])
std  = np.array([0.229, 0.224, 0.225])
for i, ax in enumerate(axes.flat):
    if i >= len(imgs): break
    img = imgs[i].permute(1,2,0).numpy()
    img = np.clip(img * std + mean, 0, 1)
    ax.imshow(img)
    ax.set_title(CLASSES[lbls[i]], fontsize=9)
    ax.axis('off')
plt.suptitle('Training Batch Samples')
plt.tight_layout()
plt.show()

## Step 6 — Build CNN Model (ResNet18 with Transfer Learning)

In [ ]:
from src.model import build_model

model = build_model(arch='resnet18', pretrained=True, freeze_backbone=False)
model = model.to(device)

# Quick sanity check
dummy = torch.randn(2, 3, 224, 224).to(device)
out   = model(dummy)
print(f'Output shape: {out.shape}')  # should be (2, 3)

## Step 7 — Train the Model

In [ ]:
from src.train import train_model

model, history = train_model(
    model       = model,
    dataloaders = loaders,
    num_epochs  = 10,
    lr          = 1e-4,
    save_dir    = '../models',
    device      = device,
)

In [ ]:
# Plot training curves inline
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'],   label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(history['train_acc'], label='Train')
axes[1].plot(history['val_acc'],   label='Val')
axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(True)
plt.tight_layout()
plt.show()

## Step 8 — Evaluate on Test Set

In [ ]:
from src.evaluate import evaluate_model

results = evaluate_model(
    model      = model,
    dataloader = loaders['test'],
    device     = device,
    save_dir   = '../outputs',
)

# Show confusion matrix
img = Image.open('../outputs/confusion_matrix.png')
plt.figure(figsize=(6, 5))
plt.imshow(img)
plt.axis('off')
plt.show()

## Step 9 — Grad-CAM Explainability

In [ ]:
import torchvision.transforms as T
from pathlib import Path
from src.gradcam import visualise_gradcam
from src.model   import get_last_conv_layer

# Pick a sample image
sample_path = next(Path(DATA_DIR).glob('**/*.png'))

transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
img_pil  = Image.open(sample_path).convert('RGB')
img_np   = np.array(img_pil.resize((224, 224)))
tensor   = transform(img_pil).unsqueeze(0)

target_layer = get_last_conv_layer(model, 'resnet18')
pred_label, confidence, probs = visualise_gradcam(
    model, target_layer, tensor, img_np,
    class_names = CLASSES,
    save_path   = '../outputs/gradcam_notebook.png',
    device      = device,
)

# Display inline
img = Image.open('../outputs/gradcam_notebook.png')
plt.figure(figsize=(14, 4))
plt.imshow(img)
plt.axis('off')
plt.title(f'Prediction: {pred_label.upper()} ({confidence:.1f}%)')
plt.show()

## Step 10 — Predict on a New Signal

In [ ]:
from src.predict import predict_from_image

result = predict_from_image(
    img_path    = str(sample_path),
    model       = model,
    device      = device,
    run_gradcam = False,
    arch        = 'resnet18',
    save_dir    = '../outputs',
)

print('\nSample Output Format:')
print(result)

## Sample Output Format

```
========================================
  EEG MENTAL STATE PREDICTION
========================================
  Predicted State : FOCUSED
  Confidence      : 87.43%

  All Probabilities:
    focused    87.43%  █████████████████
    relaxed     8.21%  █
    stressed    4.36%
========================================
```